<a href="https://colab.research.google.com/github/zeynepoztunc/aml-procedural-mistake-detection/blob/zeynep-september/error_type_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Error Type Analysis

Analyzes per-error-category performance (`error_category_recognition` task) for **MLP + Omnivore**
and **Transformer + Omnivore**, reusing the existing dataset/model/train/eval code from this repo
(`dataloader/CaptainCookStepDataset.py`, `base.py`). No models are rewritten here.

This is a separate notebook from the baseline reproduction notebook and does not modify it.

<a href="https://colab.research.google.com/github/zeynepoztunc/aml-procedural-mistake-detection/blob/zeynep-september/notebooks/error_type_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup (Colab)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!git clone --recursive --branch zeynep-september https://github.com/zeynepoztunc/aml-procedural-mistake-detection.git \
/content/code

Cloning into '/content/code'...
remote: Enumerating objects: 908, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 908 (delta 101), reused 89 (delta 87), pack-reused 786 (from 2)
Receiving objects: 100% (908/908), 96.53 MiB | 19.07 MiB/s, done.
Resolving deltas: 100% (491/491), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 1.99 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out '0e9a108be2cbcbcbd592e7418c0ab9c16232d27a'


In [4]:
%cd /content/code

/content/code


In [5]:
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime -> Change runtime type -> select a GPU."
)

DEVICE = "cuda"

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Torch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8


In [6]:
!pip install -q torcheval pyrebase4 yacs loguru wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 11.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blobfile 3.2.0 requires urllib3>=2, but you have urllib3 1.26.20 which is incompatible.


## 2. Omnivore features

Same layout the baseline notebook uses: `omnivore.zip` already contains an `omnivore/` folder, so we
extract straight into `data/video` -> `data/video/omnivore`, which is where
`CaptainCookStepDataset` expects features (`segment_features_directory="data/"` + `"video"` + backbone).

In [7]:
import os

DRIVE_BASE_PATH = "/content/drive/MyDrive/AML_Project"
FEATURES_ZIP = f"{DRIVE_BASE_PATH}/1s.zip"

CODE_DIR = "/content/code"
TEMP_DIR = "/content/temp_features"
VIDEO_DIR = f"{CODE_DIR}/data/video"

!rm -rf "{TEMP_DIR}"
!mkdir -p "{TEMP_DIR}"
!mkdir -p "{VIDEO_DIR}"

!cp "{FEATURES_ZIP}" /content/1s.zip
!unzip -q -o /content/1s.zip -d "{TEMP_DIR}"
!unzip -q -o "{TEMP_DIR}/1s/video/omnivore.zip" -d "{VIDEO_DIR}"

!rm /content/1s.zip
!rm -rf "{TEMP_DIR}"

files = os.listdir(f"{VIDEO_DIR}/omnivore")
print(f"Omnivore features ready: {len(files)} files")

Omnivore features ready: 384 files


## 3. Discover the supported error categories

We don't hardcode the category list. `CaptainCookStepDataset` defines its own
`_category_name_map` / `_error_category_name_label_map` (dataloader/CaptainCookStepDataset.py),
mapping a subset of the raw annotation tags to numeric labels used by `error_category_recognition`.
We read that mapping directly off a dataset instance, so this notebook stays correct even if the
repo's supported categories change.

In [8]:
import json
from collections import Counter

with open("annotations/annotation_json/error_annotations.json") as f:
    error_annotations = json.load(f)

tag_counts = Counter()
for recording in error_annotations:
    for step in recording.get("step_annotations", []):
        for err in step.get("errors", []):
            tag_counts[err["tag"]] += 1

print("All error tags found in the raw annotations:")
for tag, count in tag_counts.most_common():
    print(f"  {tag}: {count}")

All error tags found in the raw annotations:
  Order Error: 795
  Technique Error: 502
  Preparation Error: 410
  Measurement Error: 331
  Missing Step: 285
  Timing Error: 177
  Temperature Error: 66
  Other: 8


In [9]:
sys.path.insert(0, ".")

from types import SimpleNamespace
from constants import Constants as const
from dataloader.CaptainCookStepDataset import CaptainCookStepDataset, collate_fn
from base import fetch_model, train_epoch, test_er_model
from torch.utils.data import DataLoader

# Cheap: just loads annotations + the step split file, no feature files needed yet.
_probe_config = SimpleNamespace(backbone=const.OMNIVORE, modality="video", split=const.STEP_SPLIT, seed=1000)
_probe_dataset = CaptainCookStepDataset(_probe_config, const.TEST, const.STEP_SPLIT)

# `--error_category` values (e.g. "TechniqueError") actually supported by the code.
# These are read off the dataset class, not hardcoded.
SUPPORTED_CATEGORIES = list(_probe_dataset._category_name_map.keys())

print("Error categories supported by error_category_recognition:")
for cat in SUPPORTED_CATEGORIES:
    tag = _probe_dataset._category_name_map[cat]
    print(f"  {cat!r} -> {tag!r} (annotation count: {tag_counts.get(tag, 0)})")

print("\nNot supported by the current task code (excluded automatically):",
      sorted(set(tag_counts) - set(_probe_dataset._category_name_map.values())))

Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Error categories supported by error_category_recognition:
  'TechniqueError' -> 'Technique Error' (annotation count: 502)
  'PreparationError' -> 'Preparation Error' (annotation count: 410)
  'TemperatureError' -> 'Temperature Error' (annotation count: 66)
  'MeasurementError' -> 'Measurement Error' (annotation count: 331)
  'TimingError' -> 'Timing Error' (annotation count: 177)

Not supported by the current task code (excluded automatically): ['Missing Step', 'Order Error', 'Other']


## 4. Train + evaluate per (model, error category)

Reuses the existing building blocks directly, unmodified:
- `CaptainCookStepDataset` / `collate_fn` for data loading and per-category labeling
- `fetch_model` for the MLP / Transformer (ErFormer) architectures
- `train_epoch` / `test_er_model` for the training step and step-level metrics (Accuracy, Precision, Recall, F1, AUC)

We do **not** call `base.train_model_base` here: it writes a checkpoint to disk every epoch and logs
to wandb, but doesn't return metrics. For this sweep we just want numbers back, so `run_experiment`
below is a thin loop around the same `train_epoch` / `test_er_model` functions, using the same
train/val/test phases (`const.TRAIN` / `const.VAL` / `const.TEST`) that `base.train_step_test_step_dataset_base`
already uses elsewhere in this repo.

**Model selection (standard train/val/test workflow, no leakage):**
1. Train on the `train` split.
2. After every epoch, evaluate on the `val` split and record val AUC.
3. Keep the model weights from the epoch with the best val AUC (in memory, not written to disk).
4. After all epochs finish, restore those best-on-validation weights.
5. Evaluate that restored model **once** on the `test` split — this is the only time test data is touched.
6. Return the final test metrics, plus which epoch was selected and its validation AUC (for inspection).

In [10]:
VARIANTS = [const.MLP_VARIANT, const.TRANSFORMER_VARIANT]
NUM_EPOCHS = 5      # keep small for a quick per-category sweep; raise for more reliable numbers
BATCH_SIZE = 8
THRESHOLD = 0.6      # consistent with step-split thresholds used elsewhere in this repo


def run_experiment(variant, error_category, num_epochs=NUM_EPOCHS, threshold=THRESHOLD):
    config = SimpleNamespace(
        backbone=const.OMNIVORE,
        modality="video",
        segment_features_directory="data/",
        split=const.STEP_SPLIT,
        task_name=const.ERROR_CATEGORY_RECOGNITION,
        error_category=error_category,
        variant=variant,
        seed=1000,
        device=DEVICE,
        batch_size=BATCH_SIZE,
    )
    torch.manual_seed(config.seed)

    train_dataset = CaptainCookStepDataset(config, const.TRAIN, config.split)
    val_dataset = CaptainCookStepDataset(config, const.VAL, config.split)
    test_dataset = CaptainCookStepDataset(config, const.TEST, config.split)

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

    model = fetch_model(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.5], device=DEVICE))

    best_val_auc = -1.0
    best_epoch = None
    best_state_dict = None

    for epoch in range(1, num_epochs + 1):
        train_epoch(model, DEVICE, train_loader, optimizer, epoch, criterion)

        # Model selection uses VAL only. TEST is not touched here.
        _, _, val_metrics = test_er_model(
            model, val_loader, criterion, DEVICE, phase="val", threshold=threshold
        )
        val_auc = float(val_metrics["auc"])
        print(f"  epoch {epoch}: val AUC = {val_auc:.4f}")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch
            best_state_dict = {k: v.detach().clone() for k, v in model.state_dict().items()}

    # Restore the best-on-validation weights, then evaluate on TEST exactly once.
    model.load_state_dict(best_state_dict)
    _, _, test_metrics = test_er_model(
        model, test_loader, criterion, DEVICE, phase="test", threshold=threshold
    )

    print(f"  selected best epoch = {best_epoch} (val AUC = {best_val_auc:.4f})")
    print(f"  final TEST metrics: {test_metrics}")

    return {
        "best_epoch": best_epoch,
        "val_auc": best_val_auc,
        "accuracy": test_metrics["accuracy"],
        "precision": test_metrics["precision"],
        "recall": test_metrics["recall"],
        "f1": test_metrics["f1"],
        "auc": test_metrics["auc"],
    }

### Smoke test

Run one quick (model, category) pair first to make sure the val-selection / single-test-eval
workflow is actually wired up correctly before running the full sweep.

Check in the printed output below:
- a `val AUC = ...` line for every epoch (validation is evaluated after each epoch),
- the `selected best epoch` line (selection is based on val AUC, not test),
- exactly one `final TEST metrics` line, printed only after the best epoch was selected.

In [11]:
_smoke_metrics = run_experiment(const.MLP_VARIANT, SUPPORTED_CATEGORIES[0], num_epochs=3)

print("\nReturned dict (this is what feeds the results table):")
print(_smoke_metrics)

assert "best_epoch" in _smoke_metrics and "val_auc" in _smoke_metrics, (
    "run_experiment should report which epoch was selected and its val AUC"
)

Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


Train Epoch: 1, Progress: 468/469, Loss: 1.099043: 100%|██████████| 469/469 [00:23<00:00, 20.18it/s]
val Progress: 34385/774: 100%|██████████| 774/774 [00:05<00:00, 134.87it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.9045223207794096, 'auc': np.float64(0.519822757827492), 'pr_auc': tensor(0.0955)}
val Step Level Metrics: {'precision': 0.09090909090909091, 'recall': 0.022988505747126436, 'f1': 0.03669724770642202, 'accuracy': 0.8643410852713178, 'auc': np.float64(0.5625491475514063), 'pr_auc': tensor(0.1119)}
----------------------------------------------------------------
  epoch 1: val AUC = 0.5625


Train Epoch: 2, Progress: 468/469, Loss: 0.537461: 100%|██████████| 469/469 [00:22<00:00, 21.29it/s]
val Progress: 34385/774: 100%|██████████| 774/774 [00:05<00:00, 139.67it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.9045223207794096, 'auc': np.float64(0.5300325882826696), 'pr_auc': tensor(0.0955)}
val Step Level Metrics: {'precision': 0.375, 'recall': 0.034482758620689655, 'f1': 0.06315789473684211, 'accuracy': 0.8850129198966409, 'auc': np.float64(0.5605079556291723), 'pr_auc': tensor(0.1215)}
----------------------------------------------------------------
  epoch 2: val AUC = 0.5605


Train Epoch: 3, Progress: 468/469, Loss: 0.127661: 100%|██████████| 469/469 [00:22<00:00, 20.78it/s]
val Progress: 34385/774: 100%|██████████| 774/774 [00:05<00:00, 138.25it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.9045223207794096, 'auc': np.float64(0.45683087236393716), 'pr_auc': tensor(0.0955)}
val Step Level Metrics: {'precision': 0.2727272727272727, 'recall': 0.06896551724137931, 'f1': 0.11009174311926606, 'accuracy': 0.8746770025839793, 'auc': np.float64(0.5446803526911944), 'pr_auc': tensor(0.1235)}
----------------------------------------------------------------
  epoch 3: val AUC = 0.5447


test Progress: 42347/798: 100%|██████████| 798/798 [00:06<00:00, 128.88it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.9477885092214324, 'auc': np.float64(0.6755736229519769), 'pr_auc': tensor(0.0522)}
test Step Level Metrics: {'precision': 0.13043478260869565, 'recall': 0.04838709677419355, 'f1': 0.07058823529411765, 'accuracy': 0.9010025062656641, 'auc': np.float64(0.6956740883590462), 'pr_auc': tensor(0.0802)}
----------------------------------------------------------------
  selected best epoch = 1 (val AUC = 0.5625)
  final TEST metrics: {'precision': 0.13043478260869565, 'recall': 0.04838709677419355, 'f1': 0.07058823529411765, 'accuracy': 0.9010025062656641, 'auc': np.float64(0.6956740883590462), 'pr_auc': tensor(0.0802)}

Returned dict (this is what feeds the results table):
{'best_epoch': 1, 'val_auc': 0.5625491475514063, 'accuracy': 0.9010025062656641, 'precision': 0.13043478260869565, 'recall': 0.04838709677419355, 'f1': 0.07058823529411765,

### Full sweep

In [12]:
results = []

for variant in VARIANTS:
    for category in SUPPORTED_CATEGORIES:
        print(f"\n=== {variant} / {category} ===")
        metrics = run_experiment(variant, category)
        results.append({
            "Model": variant,
            "Error Category": category,
            "Accuracy": round(float(metrics["accuracy"]) * 100, 2),
            "Precision": round(float(metrics["precision"]) * 100, 2),
            "Recall": round(float(metrics["recall"]) * 100, 2),
            "F1": round(float(metrics["f1"]) * 100, 2),
            "AUC": round(float(metrics["auc"]) * 100, 2),
        })


=== MLP / TechniqueError ===
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


Train Epoch: 1, Progress: 468/469, Loss: 1.099043: 100%|██████████| 469/469 [00:22<00:00, 20.97it/s]
val Progress: 34385/774: 100%|██████████| 774/774 [00:05<00:00, 137.26it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.9045223207794096, 'auc': np.float64(0.519822757827492), 'pr_auc': tensor(0.0955)}
val Step Level Metrics: {'precision': 0.09090909090909091, 'recall': 0.022988505747126436, 'f1': 0.03669724770642202, 'accuracy': 0.8643410852713178, 'auc': np.float64(0.5625491475514063), 'pr_auc': tensor(0.1119)}
----------------------------------------------------------------
  epoch 1: val AUC = 0.5625


Train Epoch: 2, Progress: 468/469, Loss: 0.537461: 100%|██████████| 469/469 [00:22<00:00, 20.81it/s]
val Progress: 34385/774: 100%|██████████| 774/774 [00:05<00:00, 138.53it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.9045223207794096, 'auc': np.float64(0.5300325882826696), 'pr_auc': tensor(0.0955)}
val Step Level Metrics: {'precision': 0.375, 'recall': 0.034482758620689655, 'f1': 0.06315789473684211, 'accuracy': 0.8850129198966409, 'auc': np.float64(0.5605079556291723), 'pr_auc': tensor(0.1215)}
----------------------------------------------------------------
  epoch 2: val AUC = 0.5605


...

----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.8549949105714701, 'auc': np.float64(0.5730611070362982), 'pr_auc': tensor(0.1450)}
val Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.9366925064599483, 'auc': np.float64(0.5872602589020499), 'pr_auc': tensor(0.0478)}
----------------------------------------------------------------
  epoch 4: val AUC = 0.5873


Train Epoch: 5, Progress: 468/469, Loss: 0.028319: 100%|██████████| 469/469 [00:31<00:00, 14.95it/s]
val Progress: 34385/774: 100%|██████████| 774/774 [00:06<00:00, 112.96it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.8549949105714701, 'auc': np.float64(0.6586200946308973), 'pr_auc': tensor(0.1450)}
val Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.9496124031007752, 'auc': np.float64(0.7445084161502072), 'pr_auc': tensor(0.0478)}
----------------------------------------------------------------
  epoch 5: val AUC = 0.7445


test Progress: 42347/798: 100%|██████████| 798/798 [00:07<00:00, 100.75it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.9330531088388788, 'auc': np.float64(0.6030706274395955), 'pr_auc': tensor(0.0669)}
test Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.9523809523809523, 'auc': np.float64(0.7250923929781337), 'pr_auc': tensor(0.0426)}
----------------------------------------------------------------
  selected best epoch = 5 (val AUC = 0.7445)
  final TEST metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.9523809523809523, 'auc': np.float64(0.7250923929781337), 'pr_auc': tensor(0.0426)}


## 5. Results

In [13]:
import pandas as pd

df = pd.DataFrame(results)[["Model", "Error Category", "Accuracy", "Precision", "Recall", "F1", "AUC"]]
df = df.sort_values(["Error Category", "Model"]).reset_index(drop=True)
df

,Model,Error Category,Accuracy,Precision,Recall,F1,AUC
0,MLP,MeasurementError,92.36,17.24,11.90,14.08,75.94
1,Transformer,MeasurementError,77.44,12.90,57.14,21.05,71.65
2,MLP,PreparationError,92.11,29.41,20.41,24.10,69.49
3,Transformer,PreparationError,90.73,3.70,2.04,2.63,46.58
4,MLP,TechniqueError,90.10,13.04,4.84,7.06,69.57
5,Transformer,TechniqueError,91.23,10.00,1.61,2.78,60.68
6,MLP,TemperatureError,98.37,0.00,0.00,0.00,69.67
7,Transformer,TemperatureError,98.75,0.00,0.00,0.00,70.73
8,MLP,TimingError,95.49,0.00,0.00,0.00,81.48
9,Transformer,TimingError,95.24,0.00,0.00,0.00,72.51
